# Step 14: Leakage-Safe Feature Engineering

This step creates the internal features used by the seasonal naive, XGBoost and LSTM comparisons.

The modelling dataset contains 800 item-store series and 1,552,800 daily observations. Every sales-derived feature is calculated using information from earlier dates only.

The five feature groups are:

1. Lag features: lag_7, lag_14, lag_28 and lag_56
2. Rolling features: rolling_mean_7, rolling_mean_28, rolling_std_7 and rolling_std_28
3. Calendar features: weekday, month, year and is_weekend
4. Event and SNAP features: event_flag, event_type_encoded and snap_flag
5. Price features: sell_price, price_change and price_vs_28d_avg

The same-day sales value is used only as the prediction target. It is never included in a lag or rolling feature.

The current calendar, event, SNAP and scheduled selling-price variables are treated as information known at the forecast date. No future sales observations are used.

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

np.random.seed(42)

project_root = Path("..")

subset_data_path = project_root / "data" / "subset"
processed_data_path = project_root / "data" / "processed"
tables_path = project_root / "outputs" / "tables" / "step14"

tables_path.mkdir(parents=True, exist_ok=True)

input_file = subset_data_path / "selected_m5_data.parquet"
output_file = processed_data_path / "modelling_dataset.parquet"

print("Input:", input_file)
print("Output:", output_file)

Input: ..\data\subset\selected_m5_data.parquet
Output: ..\data\processed\modelling_dataset.parquet


In [2]:
model_data = pd.read_parquet(input_file)

model_data["date"] = pd.to_datetime(
    model_data["date"]
)

model_data = model_data.sort_values(
    ["id", "date"]
).reset_index(drop=True)

initial_rows = len(model_data)
initial_series = model_data["id"].nunique()

rows_per_series = (
    model_data
    .groupby("id", observed=True)
    .size()
)

print("Rows:", initial_rows)
print("Unique series:", initial_series)
print("Minimum rows per series:", rows_per_series.min())
print("Maximum rows per series:", rows_per_series.max())

print(
    "Duplicate id-date rows:",
    model_data.duplicated(
        subset=["id", "date"]
    ).sum()
)

print(
    "Missing demand types:",
    model_data["demand_type"].isna().sum()
)

print(
    "Date range:",
    model_data["date"].min(),
    "to",
    model_data["date"].max()
)

Rows: 1552800
Unique series: 800
Minimum rows per series: 1941
Maximum rows per series: 1941
Duplicate id-date rows: 0
Missing demand types: 0
Date range: 2011-01-29 00:00:00 to 2016-05-22 00:00:00


In [3]:
model_data["sales"] = model_data["sales"].astype(
    "int32"
)

model_data["wm_yr_wk"] = model_data[
    "wm_yr_wk"
].astype("int32")

model_data["wday"] = model_data[
    "wday"
].astype("int8")

model_data["month"] = model_data[
    "month"
].astype("int8")

model_data["year"] = model_data[
    "year"
].astype("int16")

model_data["snap"] = model_data[
    "snap"
].astype("int8")

model_data["sell_price"] = model_data[
    "sell_price"
].astype("float32")

category_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "demand_type"
]

for column in category_columns:
    model_data[column] = model_data[
        column
    ].astype("category")

print(
    "Memory usage MB:",
    round(
        model_data.memory_usage(
            deep=True
        ).sum() / (1024 ** 2),
        2
    )
)

Memory usage MB: 400.56


In [4]:
lag_periods = [7, 14, 28, 56]

sales_group = model_data.groupby(
    "id",
    observed=True,
    sort=False
)["sales"]

for lag in lag_periods:
    model_data[f"lag_{lag}"] = (
        sales_group
        .shift(lag)
        .astype("float32")
    )

model_data[
    [
        "id",
        "date",
        "sales",
        "lag_7",
        "lag_14",
        "lag_28",
        "lag_56"
    ]
].head(60)

,id,date,sales,lag_7,lag_14,lag_28,lag_56
0,FOODS_1_004_WI_1_evaluation,2011-01-29,0,NaN,NaN,NaN,NaN
1,FOODS_1_004_WI_1_evaluation,2011-01-30,0,NaN,NaN,NaN,NaN
2,FOODS_1_004_WI_1_evaluation,2011-01-31,0,NaN,NaN,NaN,NaN
3,FOODS_1_004_WI_1_evaluation,2011-02-01,0,NaN,NaN,NaN,NaN
4,FOODS_1_004_WI_1_evaluation,2011-02-02,0,NaN,NaN,NaN,NaN
5,FOODS_1_004_WI_1_evaluation,2011-02-03,0,NaN,NaN,NaN,NaN
6,FOODS_1_004_WI_1_evaluation,2011-02-04,0,NaN,NaN,NaN,NaN
7,FOODS_1_004_WI_1_evaluation,2011-02-05,0,0.0,NaN,NaN,NaN
8,FOODS_1_004_WI_1_evaluation,2011-02-06,0,0.0,NaN,NaN,NaN
9,FOODS_1_004_WI_1_evaluation,2011-02-07,0,0.0,NaN,NaN,NaN


In [8]:
past_sales = (
    model_data
    .groupby(
        "id",
        observed=True,
        sort=False
    )["sales"]
    .shift(1)
    .astype("float32")
)

for window in [7, 28]:
    grouped_past_sales = past_sales.groupby(
        model_data["id"],
        observed=True,
        sort=False
    )
    
    rolling_mean = (
        grouped_past_sales
        .rolling(
            window=window,
            min_periods=window
        )
        .mean()
        .reset_index(
            level=0,
            drop=True
        )
        .sort_index()
    )
    
    grouped_past_sales = past_sales.groupby(
        model_data["id"],
        observed=True,
        sort=False
    )
    
    rolling_std = (
        grouped_past_sales
        .rolling(
            window=window,
            min_periods=window
        )
        .std(ddof=0)
        .reset_index(
            level=0,
            drop=True
        )
        .sort_index()
    )
    
    model_data[
        f"rolling_mean_{window}"
    ] = rolling_mean.astype("float32")
    
    model_data[
        f"rolling_std_{window}"
    ] = rolling_std.astype("float32")

del past_sales
gc.collect()

416

In [10]:
# Remove duplicate columns created by rerunning the rename cell
duplicate_columns = model_data.columns[
    model_data.columns.duplicated()
].tolist()

print("Duplicate columns found:", duplicate_columns)

if duplicate_columns:
    model_data = model_data.loc[
        :,
        ~model_data.columns.duplicated(keep="first")
    ].copy()

# M5 weekday mapping: wday 1 = Saturday
weekday_name_mapping = {
    1: "Saturday",
    2: "Sunday",
    3: "Monday",
    4: "Tuesday",
    5: "Wednesday",
    6: "Thursday",
    7: "Friday"
}

model_data["weekday_name"] = (
    model_data["wday"]
    .map(weekday_name_mapping)
    .astype("category")
)

model_data["weekday"] = (
    model_data["wday"]
    .astype("int8")
)

# Saturday and Sunday are wday 1 and 2 in M5
model_data["is_weekend"] = (
    model_data["wday"]
    .isin([1, 2])
    .astype("int8")
)

print(
    "Duplicate columns remaining:",
    model_data.columns.duplicated().sum()
)

print(
    "Weekday values:",
    sorted(model_data["weekday"].unique())
)

print(
    "Weekend rows:",
    int(model_data["is_weekend"].sum())
)

print(
    "Weekend percentage:",
    round(
        model_data["is_weekend"].mean() * 100,
        3
    )
)

Duplicate columns found: ['weekday_name', 'weekday_name']
Duplicate columns remaining: 0
Weekday values: [np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5), np.int8(6), np.int8(7)]
Weekend rows: 444800
Weekend percentage: 28.645


In [11]:
event_types_found = set(
    pd.concat(
        [
            model_data["event_type_1"],
            model_data["event_type_2"]
        ]
    )
    .dropna()
    .unique()
)

event_type_mapping = {
    "Cultural": 1,
    "National": 2,
    "Religious": 3,
    "Sporting": 4
}

unexpected_event_types = (
    event_types_found
    - set(event_type_mapping.keys())
)

print("Event types found:", event_types_found)
print(
    "Unexpected event types:",
    unexpected_event_types
)

if unexpected_event_types:
    raise ValueError(
        "Unexpected event type found."
    )

Event types found: {'Religious', 'Sporting', 'National', 'Cultural'}
Unexpected event types: set()


In [12]:
model_data["event_flag"] = (
    model_data["event_name_1"].notna()
    | model_data["event_name_2"].notna()
).astype("int8")

primary_event_code = (
    model_data["event_type_1"]
    .map(event_type_mapping)
    .fillna(0)
    .astype("int8")
)

model_data["event_type_encoded"] = np.where(
    model_data["event_type_2"].notna(),
    5,
    primary_event_code
).astype("int8")

model_data["snap_flag"] = model_data[
    "snap"
].astype("int8")

print(
    "Event rows:",
    int(model_data["event_flag"].sum())
)

print(
    "SNAP rows:",
    int(model_data["snap_flag"].sum())
)

print()
print(
    model_data[
        "event_type_encoded"
    ].value_counts().sort_index()
)

Event rows: 126400
SNAP rows: 512000

event_type_encoded
0    1426400
1      28800
2      40800
3      41600
4      12000
5       3200
Name: count, dtype: int64


In [13]:
maximum_weekly_prices = (
    model_data
    .groupby(
        ["id", "wm_yr_wk"],
        observed=True
    )["sell_price"]
    .nunique(dropna=True)
    .max()
)

print(
    "Maximum distinct prices "
    "within one item-week:",
    maximum_weekly_prices
)

if maximum_weekly_prices > 1:
    raise ValueError(
        "More than one selling price "
        "was found within an item-week."
    )

Maximum distinct prices within one item-week: 1


In [14]:
weekly_price = (
    model_data[
        [
            "id",
            "wm_yr_wk",
            "sell_price"
        ]
    ]
    .drop_duplicates(
        subset=[
            "id",
            "wm_yr_wk"
        ]
    )
    .sort_values(
        ["id", "wm_yr_wk"]
    )
    .reset_index(drop=True)
)

weekly_price["previous_week_price"] = (
    weekly_price
    .groupby(
        "id",
        observed=True,
        sort=False
    )["sell_price"]
    .shift(1)
)

weekly_price["price_change"] = np.where(
    (
        weekly_price[
            "sell_price"
        ].notna()
    )
    & (
        weekly_price[
            "previous_week_price"
        ].gt(0)
    ),
    (
        weekly_price["sell_price"]
        / weekly_price[
            "previous_week_price"
        ]
    ) - 1,
    np.nan
)

weekly_price["price_change"] = weekly_price[
    "price_change"
].astype("float32")

model_data["_row_order"] = np.arange(
    len(model_data)
)

model_data = model_data.merge(
    weekly_price[
        [
            "id",
            "wm_yr_wk",
            "price_change"
        ]
    ],
    on=[
        "id",
        "wm_yr_wk"
    ],
    how="left",
    validate="many_to_one",
    sort=False
)

model_data = (
    model_data
    .sort_values("_row_order")
    .drop(columns="_row_order")
    .reset_index(drop=True)
)

print(
    "Non-missing price changes:",
    model_data["price_change"]
    .notna()
    .sum()
)

print(
    "Price-change missing percentage:",
    round(
        model_data["price_change"]
        .isna()
        .mean()
        * 100,
        3
    )
)

Non-missing price changes: 1389308
Price-change missing percentage: 10.529


In [15]:
past_price = (
    model_data
    .groupby(
        "id",
        observed=True,
        sort=False
    )["sell_price"]
    .shift(1)
)

past_price_28d_average = (
    past_price
    .groupby(
        model_data["id"],
        observed=True,
        sort=False
    )
    .rolling(
        window=28,
        min_periods=7
    )
    .mean()
    .reset_index(
        level=0,
        drop=True
    )
    .sort_index()
)

model_data["price_vs_28d_avg"] = np.where(
    (
        model_data["sell_price"].notna()
    )
    & (
        past_price_28d_average.gt(0)
    ),
    (
        model_data["sell_price"]
        / past_price_28d_average
    ),
    np.nan
).astype("float32")

model_data["price_vs_28d_avg"] = (
    model_data["price_vs_28d_avg"]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

del past_price
del past_price_28d_average
del weekly_price

gc.collect()

1578

In [16]:
feature_groups = {
    "lag": [
        "lag_7",
        "lag_14",
        "lag_28",
        "lag_56"
    ],
    "rolling": [
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ],
    "calendar": [
        "weekday",
        "month",
        "year",
        "is_weekend"
    ],
    "event_snap": [
        "event_flag",
        "event_type_encoded",
        "snap_flag"
    ],
    "price": [
        "sell_price",
        "price_change",
        "price_vs_28d_avg"
    ]
}

all_model_features = [
    feature
    for group_features
    in feature_groups.values()
    for feature in group_features
]

print(
    "Number of model features:",
    len(all_model_features)
)

for group_name, features in feature_groups.items():
    print(group_name, ":", features)

Number of model features: 18
lag : ['lag_7', 'lag_14', 'lag_28', 'lag_56']
rolling : ['rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28']
calendar : ['weekday', 'month', 'year', 'is_weekend']
event_snap : ['event_flag', 'event_type_encoded', 'snap_flag']
price : ['sell_price', 'price_change', 'price_vs_28d_avg']


In [17]:
lag_validation = {}

for lag in lag_periods:
    expected_lag = (
        model_data
        .groupby(
            "id",
            observed=True,
            sort=False
        )["sales"]
        .shift(lag)
    )
    
    lag_validation[
        f"lag_{lag}"
    ] = np.allclose(
        model_data[f"lag_{lag}"],
        expected_lag,
        equal_nan=True
    )
    
    del expected_lag

lag_validation

{'lag_7': True, 'lag_14': True, 'lag_28': True, 'lag_56': True}

In [18]:
validation_ids = (
    model_data["id"]
    .drop_duplicates()
    .sample(
        n=8,
        random_state=42
    )
    .tolist()
)

rolling_validation_records = []

for series_id in validation_ids:
    series_check = (
        model_data.loc[
            model_data["id"] == series_id
        ]
        .sort_values("date")
        .copy()
    )
    
    expected_mean_7 = (
        series_check["sales"]
        .shift(1)
        .rolling(
            window=7,
            min_periods=7
        )
        .mean()
    )
    
    expected_mean_28 = (
        series_check["sales"]
        .shift(1)
        .rolling(
            window=28,
            min_periods=28
        )
        .mean()
    )
    
    expected_std_7 = (
        series_check["sales"]
        .shift(1)
        .rolling(
            window=7,
            min_periods=7
        )
        .std(ddof=0)
    )
    
    expected_std_28 = (
        series_check["sales"]
        .shift(1)
        .rolling(
            window=28,
            min_periods=28
        )
        .std(ddof=0)
    )
    
    rolling_validation_records.append({
        "id": str(series_id),
        "rolling_mean_7_valid": np.allclose(
            series_check["rolling_mean_7"],
            expected_mean_7,
            equal_nan=True
        ),
        "rolling_mean_28_valid": np.allclose(
            series_check["rolling_mean_28"],
            expected_mean_28,
            equal_nan=True
        ),
        "rolling_std_7_valid": np.allclose(
            series_check["rolling_std_7"],
            expected_std_7,
            equal_nan=True
        ),
        "rolling_std_28_valid": np.allclose(
            series_check["rolling_std_28"],
            expected_std_28,
            equal_nan=True
        )
    })

rolling_validation = pd.DataFrame(
    rolling_validation_records
)

rolling_validation

,id,rolling_mean_7_valid,rolling_mean_28_valid,rolling_std_7_valid,rolling_std_28_valid
0,HOUSEHOLD_1_389_TX_1_evaluation,True,True,True,True
1,HOUSEHOLD_1_303_CA_3_evaluation,True,True,True,True
2,FOODS_2_020_TX_3_evaluation,True,True,True,True
3,HOBBIES_1_249_CA_3_evaluation,True,True,True,True
4,FOODS_2_043_CA_2_evaluation,True,True,True,True
5,HOUSEHOLD_1_114_CA_2_evaluation,True,True,True,True
6,FOODS_3_501_TX_2_evaluation,True,True,True,True
7,HOBBIES_1_043_CA_3_evaluation,True,True,True,True


In [19]:
rolling_check_columns = [
    "rolling_mean_7_valid",
    "rolling_mean_28_valid",
    "rolling_std_7_valid",
    "rolling_std_28_valid"
]

print(
    "All rolling checks valid:",
    rolling_validation[
        rolling_check_columns
    ].all().all()
)

All rolling checks valid: True


In [20]:
feature_missingness_by_type = (
    model_data[
        all_model_features
    ]
    .isna()
    .groupby(
        model_data["demand_type"],
        observed=True
    )
    .mean()
    * 100
).round(3)

feature_missingness_by_type

,lag_7,lag_14,lag_28,lag_56,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28,weekday,month,year,is_weekend,event_flag,event_type_encoded,snap_flag,sell_price,price_change,price_vs_28d_avg
demand_type,,,,,,,,,,,,,,,,,,
smooth,0.361,0.721,1.443,2.885,0.361,1.443,0.361,1.443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.642,1.003,1.003
erratic,0.361,0.721,1.443,2.885,0.361,1.443,0.361,1.443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.399,0.759,0.759
intermittent,0.361,0.721,1.443,2.885,0.361,1.443,0.361,1.443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.742,21.103,21.103
lumpy,0.361,0.721,1.443,2.885,0.361,1.443,0.361,1.443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.890,19.251,19.251


In [21]:
feature_missingness_by_type.to_csv(
    tables_path
    / "step14_feature_missingness_by_demand_type.csv"
)

In [22]:
sales_history_features = (
    feature_groups["lag"]
    + feature_groups["rolling"]
)

sales_features_complete = (
    model_data[
        sales_history_features
    ]
    .notna()
    .all(axis=1)
)

complete_rows_by_type = (
    model_data
    .assign(
        sales_features_complete=
        sales_features_complete
    )
    .groupby(
        "demand_type",
        observed=True
    )
    .agg(
        total_rows=("id", "size"),
        complete_sales_feature_rows=(
            "sales_features_complete",
            "sum"
        )
    )
)

complete_rows_by_type[
    "complete_percentage"
] = (
    complete_rows_by_type[
        "complete_sales_feature_rows"
    ]
    / complete_rows_by_type[
        "total_rows"
    ]
    * 100
).round(3)

complete_rows_by_type

,total_rows,complete_sales_feature_rows,complete_percentage
demand_type,,,
smooth,388200,377000,97.115
erratic,388200,377000,97.115
intermittent,388200,377000,97.115
lumpy,388200,377000,97.115


In [23]:
expected_complete_rows = (
    800 * (1941 - 56)
)

actual_complete_rows = int(
    sales_features_complete.sum()
)

print(
    "Expected complete sales-feature rows:",
    expected_complete_rows
)

print(
    "Actual complete sales-feature rows:",
    actual_complete_rows
)

print(
    "Complete-row match:",
    actual_complete_rows
    == expected_complete_rows
)

Expected complete sales-feature rows: 1508000
Actual complete sales-feature rows: 1508000
Complete-row match: True


In [24]:
feature_dictionary = pd.DataFrame([
    {
        "feature_group": "Lag",
        "feature": "lag_7",
        "definition": "Sales exactly 7 days earlier",
        "leakage_control": "grouped shift by 7 dates"
    },
    {
        "feature_group": "Lag",
        "feature": "lag_14",
        "definition": "Sales exactly 14 days earlier",
        "leakage_control": "grouped shift by 14 dates"
    },
    {
        "feature_group": "Lag",
        "feature": "lag_28",
        "definition": "Sales exactly 28 days earlier",
        "leakage_control": "grouped shift by 28 dates"
    },
    {
        "feature_group": "Lag",
        "feature": "lag_56",
        "definition": "Sales exactly 56 days earlier",
        "leakage_control": "grouped shift by 56 dates"
    },
    {
        "feature_group": "Rolling",
        "feature": "rolling_mean_7",
        "definition": "Mean sales over the previous 7 days",
        "leakage_control": "sales shifted by 1 day before rolling"
    },
    {
        "feature_group": "Rolling",
        "feature": "rolling_mean_28",
        "definition": "Mean sales over the previous 28 days",
        "leakage_control": "sales shifted by 1 day before rolling"
    },
    {
        "feature_group": "Rolling",
        "feature": "rolling_std_7",
        "definition": "Sales standard deviation over the previous 7 days",
        "leakage_control": "sales shifted by 1 day before rolling"
    },
    {
        "feature_group": "Rolling",
        "feature": "rolling_std_28",
        "definition": "Sales standard deviation over the previous 28 days",
        "leakage_control": "sales shifted by 1 day before rolling"
    },
    {
        "feature_group": "Calendar",
        "feature": "weekday",
        "definition": "M5 weekday number from 1 to 7",
        "leakage_control": "known calendar value"
    },
    {
        "feature_group": "Calendar",
        "feature": "month",
        "definition": "Calendar month from 1 to 12",
        "leakage_control": "known calendar value"
    },
    {
        "feature_group": "Calendar",
        "feature": "year",
        "definition": "Calendar year",
        "leakage_control": "known calendar value"
    },
    {
        "feature_group": "Calendar",
        "feature": "is_weekend",
        "definition": "1 for Saturday or Sunday, otherwise 0",
        "leakage_control": "known calendar value"
    },
    {
        "feature_group": "Event and SNAP",
        "feature": "event_flag",
        "definition": "1 when at least one M5 event is listed",
        "leakage_control": "known calendar value"
    },
    {
        "feature_group": "Event and SNAP",
        "feature": "event_type_encoded",
        "definition": "Numeric encoding of the listed event type",
        "leakage_control": "known calendar value"
    },
    {
        "feature_group": "Event and SNAP",
        "feature": "snap_flag",
        "definition": "State-specific SNAP activity indicator",
        "leakage_control": "known calendar value"
    },
    {
        "feature_group": "Price",
        "feature": "sell_price",
        "definition": "Scheduled weekly selling price",
        "leakage_control": "current scheduled price, not future sales"
    },
    {
        "feature_group": "Price",
        "feature": "price_change",
        "definition": "Proportional change from the previous listed week",
        "leakage_control": "previous weekly price used as comparison"
    },
    {
        "feature_group": "Price",
        "feature": "price_vs_28d_avg",
        "definition": "Current price divided by the prior 28-day average",
        "leakage_control": "price average shifted by 1 day"
    }
])

feature_dictionary.to_csv(
    tables_path
    / "step14_feature_dictionary.csv",
    index=False
)

feature_dictionary

,feature_group,feature,definition,leakage_control
0,Lag,lag_7,Sales exactly 7 days earlier,grouped shift by 7 dates
1,Lag,lag_14,Sales exactly 14 days earlier,grouped shift by 14 dates
2,Lag,lag_28,Sales exactly 28 days earlier,grouped shift by 28 dates
3,Lag,lag_56,Sales exactly 56 days earlier,grouped shift by 56 dates
4,Rolling,rolling_mean_7,Mean sales over the previous 7 days,sales shifted by 1 day before rolling
5,Rolling,rolling_mean_28,Mean sales over the previous 28 days,sales shifted by 1 day before rolling
6,Rolling,rolling_std_7,Sales standard deviation over the previous 7 days,sales shifted by 1 day before rolling
7,Rolling,rolling_std_28,Sales standard deviation over the previous 28 ...,sales shifted by 1 day before rolling
8,Calendar,weekday,M5 weekday number from 1 to 7,known calendar value
9,Calendar,month,Calendar month from 1 to 12,known calendar value


In [25]:
validation_summary = pd.DataFrame([
    {
        "check": "Input rows preserved",
        "actual": len(model_data),
        "expected": initial_rows,
        "passed": len(model_data) == initial_rows
    },
    {
        "check": "Series preserved",
        "actual": model_data["id"].nunique(),
        "expected": initial_series,
        "passed": (
            model_data["id"].nunique()
            == initial_series
        )
    },
    {
        "check": "Duplicate id-date rows",
        "actual": model_data.duplicated(
            subset=["id", "date"]
        ).sum(),
        "expected": 0,
        "passed": (
            model_data.duplicated(
                subset=["id", "date"]
            ).sum()
            == 0
        )
    },
    {
        "check": "Missing demand types",
        "actual": model_data[
            "demand_type"
        ].isna().sum(),
        "expected": 0,
        "passed": (
            model_data[
                "demand_type"
            ].isna().sum()
            == 0
        )
    },
    {
        "check": "Lag calculations valid",
        "actual": all(
            lag_validation.values()
        ),
        "expected": True,
        "passed": all(
            lag_validation.values()
        )
    },
    {
        "check": "Rolling calculations valid",
        "actual": rolling_validation[
            rolling_check_columns
        ].all().all(),
        "expected": True,
        "passed": rolling_validation[
            rolling_check_columns
        ].all().all()
    },
    {
        "check": "Complete sales-feature rows",
        "actual": actual_complete_rows,
        "expected": expected_complete_rows,
        "passed": (
            actual_complete_rows
            == expected_complete_rows
        )
    }
])

validation_summary

,check,actual,expected,passed
0,Input rows preserved,1552800,1552800,True
1,Series preserved,800,800,True
2,Duplicate id-date rows,0,0,True
3,Missing demand types,0,0,True
4,Lag calculations valid,True,True,True
5,Rolling calculations valid,True,True,True
6,Complete sales-feature rows,1508000,1508000,True


In [26]:
validation_summary.to_csv(
    tables_path
    / "step14_validation_summary.csv",
    index=False
)

rolling_validation.to_csv(
    tables_path
    / "step14_rolling_validation.csv",
    index=False
)

complete_rows_by_type.to_csv(
    tables_path
    / "step14_complete_rows_by_demand_type.csv"
)

print(
    "All validation checks passed:",
    validation_summary["passed"].all()
)

All validation checks passed: True


In [27]:
model_data.to_parquet(
    output_file,
    index=False,
    engine="pyarrow",
    compression="zstd"
)

print("Saved:", output_file)

Saved: ..\data\processed\modelling_dataset.parquet


In [28]:
saved_parquet = pq.ParquetFile(
    output_file
)

saved_rows = (
    saved_parquet.metadata.num_rows
)

saved_columns = (
    saved_parquet.metadata.num_columns
)

print("Saved rows:", saved_rows)
print("Saved columns:", saved_columns)

print(
    "Saved row match:",
    saved_rows == initial_rows
)

print(
    "Saved file size MB:",
    round(
        output_file.stat().st_size
        / (1024 ** 2),
        3
    )
)

Saved rows: 1552800
Saved columns: 39
Saved row match: True
Saved file size MB: 10.135


In [29]:
feature_group_records = []

for group_name, features in feature_groups.items():
    for feature in features:
        feature_group_records.append({
            "feature_group": group_name,
            "feature": feature
        })

feature_group_table = pd.DataFrame(
    feature_group_records
)

feature_group_table.to_csv(
    tables_path
    / "step14_feature_groups.csv",
    index=False
)

feature_group_table

,feature_group,feature
0,lag,lag_7
1,lag,lag_14
2,lag,lag_28
3,lag,lag_56
4,rolling,rolling_mean_7
5,rolling,rolling_mean_28
6,rolling,rolling_std_7
7,rolling,rolling_std_28
8,calendar,weekday
9,calendar,month


## Step 14 Interpretation

### 14.1 Feature construction and data integrity

Step 14 created 18 modelling features across 5 feature groups for 1,552,800 daily observations and 800 item-store series. The final groups contain 4 lag features, 4 rolling features, 4 calendar features, 3 event and SNAP features, and 3 price features.

A total of 0 rows and 0 series were lost during feature construction. The saved modelling dataset contains 1,552,800 rows and 39 columns, with a compressed file size of 10.135 MB.

All 4 demand types retained 388,200 rows each. Smooth, erratic, intermittent and lumpy demand therefore continue to represent 25.0% each of the balanced modelling dataset.

The final validation found 0 duplicate item-date rows and 0 missing demand-type values. All 4 lag checks and all 4 rolling-feature checks returned `True`.

### 14.2 Lag and rolling features

The 4 lag features represent sales from 7, 14, 28 and 56 days earlier. Full-dataset validation confirmed that `lag_7`, `lag_14`, `lag_28` and `lag_56` matched the expected shifted sales values for all 1,552,800 rows.

The 4 rolling features were calculated after shifting sales by 1 day. This prevented the current target value from entering its own rolling mean or standard deviation.

Rolling calculations were checked manually for 8 selected item-store series. All 32 comparisons passed: 8 series multiplied by 4 rolling features.

Feature availability was identical across the 4 demand types because each selected series contained the same 1,941-day history. The missing rate was 0.361% for `lag_7`, `rolling_mean_7` and `rolling_std_7`, while the missing rate increased to 2.885% for `lag_56`.

Each demand type retained 377,000 rows with complete lag and rolling information. This equals 97.115% of the 388,200 rows available for each demand type.

Across the full modelling dataset, 1,508,000 rows had complete sales-history features. The remaining 44,800 rows represent the first 56 dates of each of the 800 series.

The first 56 dates were retained in the saved dataset for traceability, but they will not be used as training observations when all 8 sales-history features are required.

### 14.3 Calendar, event and SNAP features

The calendar group contains 4 features: weekday, month, year and weekend status. The weekday variable contains values from 1 to 7, and the weekend flag marks Saturday and Sunday using M5 weekday codes 1 and 2.

Weekend dates accounted for 444,800 of the 1,552,800 rows, equal to 28.645%. These calendar values are available for smooth, erratic, intermittent and lumpy demand before the forecast date.

At least 1 event was listed for 126,400 rows, while SNAP was active for 512,000 rows. The event encoding produced 1,426,400 rows with no event and 126,400 rows with at least 1 event.

Religious events accounted for 41,600 rows, national events for 40,800 rows, cultural events for 28,800 rows, sporting events for 12,000 rows, and dates containing a second listed event for 3,200 rows.

A total of 0 unexpected event types were found. The 4 expected M5 event categories were Cultural, National, Religious and Sporting.

### 14.4 Price features

The price group contains 3 variables: current selling price, proportional change from the previous listed week, and current price relative to the previous 28-day average.

The maximum number of distinct prices within an item-week was 1. This confirms that the weekly price table was merged consistently for all 800 selected series.

Price availability differed sharply across demand types. Missing current prices represented 0.642% of smooth rows and 0.399% of erratic rows, compared with 20.742% of intermittent rows and 18.890% of lumpy rows.

The current-price missingness gap between intermittent and erratic demand was 20.343 percentage points. This matches the Step 13 finding that intermittent products had much lower historical price coverage.

Missing `price_change` values accounted for 1.003% of smooth rows, 0.759% of erratic rows, 21.103% of intermittent rows and 19.251% of lumpy rows. The extra missing values occur when no valid previous listed-week price is available.

The missing rates for `price_vs_28d_avg` were also 1.003% for smooth, 0.759% for erratic, 21.103% for intermittent and 19.251% for lumpy demand.

The 3 price variables were left as missing where valid historical information was unavailable. Backward-filling the 21.103% missing price-change values for intermittent demand would introduce information from later product periods.

### 14.5 Leakage controls

Every sales-derived feature uses an earlier sales observation. The shortest sales lag is 7 days, and every rolling statistic excludes the current sales value through a 1-day shift.

The `price_change` variable compares the current scheduled price with the previous listed-week price. The `price_vs_28d_avg` denominator uses prices from earlier dates and excludes the current date through a 1-day shift.

Calendar, event, SNAP and current scheduled price variables are treated as information available at the forecast date. A total of 0 future sales observations were used in the 18 model features.

### 14.6 Implications for modelling

The final modelling dataset provides 1,508,000 rows with complete lag and rolling features. Each demand type contributes 377,000 eligible rows before the 5 walk-forward validation windows are applied.

Price missingness will remain visible during XGBoost training rather than being used as a reason to remove observations. Dropping rows with missing price values would remove about 21.103% of intermittent observations but only 0.759% of erratic observations.

This unequal loss would distort the balanced demand-type comparison. XGBoost will therefore receive missing values for the 3 price features while the 8 lag and rolling features must be complete.

Step 14 only confirms that the 18 features are available and leakage-safe. Whether each feature group improves WRMSSE for smooth, erratic, intermittent or lumpy demand will be tested through the Step 17 ablation study.


### Surprise Log

The main surprise in Step 14 was the difference in price-feature availability across demand types. Current selling price was missing for 20.742% of intermittent rows but only 0.399% of erratic rows, a gap of 20.343 percentage points.

Removing every row with a missing price feature would therefore affect intermittent demand much more heavily than erratic demand. The missing price values will remain in the modelling dataset so that all 4 demand types keep the same 377,000 rows with complete sales-history features.


Complete leakage-safe feature engineering for demand-type models